# Chess Module

## Dependencies

Explanatory comments in the following block are added before the command. 

In [144]:
# A python package to store binary files. 
# !pip install dill

# A python package for embedding HTML widgets in Jupyter notebooks
# !pip install ipywidgets

# A python package for embedding HTML widgets in Jupyter lab (different from notebook)
# !pip install jupyterlab_widgets

# A python package for embedding HTML widgets in Jupyter lab (different from notebook)
# Use this if the previous command fails.
# %conda install jupyterlab_widgets

# Chess Package
# !pip install python-chess

# %pip install numpy dill

## Starting Positions

Sample starting positions to test your AI agent. Some of these games have high branch factor which can cause a very slow search when searching deeper than 4.

In [145]:
# Openings to investigate
# CAUTION: Will take time on higher depths ( > 4)
ClosedSicilianDefence = 'rnbqkbnr/pp1ppppp/8/2p5/4P3/2N5/PPPP1PPP/R1BQKBNR b KQkq - 1 2'
ViennaOpenning = 'rnbqkbnr/pppp1ppp/8/4p3/4P3/2N5/PPPP1PPP/R1BQKBNR b KQkq - 1 2'
NimzoIndianDefence = 'rnbqk2r/pppp1ppp/4pn2/8/1bPP4/2N5/PP2PPPP/R1BQKBNR w KQkq - 2 4'
SlavDefence = 'rnbqkbnr/pp2pppp/2p5/3p4/2PP4/8/PP2PPPP/RNBQKBNR w KQkq - 0 3'
QueenzGambit = 'rnbqkbnr/ppp1pppp/8/3p4/2PP4/8/PP2PPPP/RNBQKBNR b KQkq c3 0 2'
RuyLopez = 'r1bqkbnr/pppp1ppp/2n5/1B2p3/4P3/5N2/PPPP1PPP/RNBQK2R b KQkq - 3 3'

WM5Position = "2nrkbn1/pp1bp3/2qprp1p/6p1/4P2P/2NP1N2/PPP2PP1/R1BQKB1R w KQ - 0 1"
BM5Position = "2kr4/1b2b1p1/p2pp3/1p6/5B2/2P1NqP1/PP3PQK/8 b - - 0 1"
BM6Position = "r1b1kr2/bp2qp2/p2p2P1/2n1p3/2Q1P1P1/1P1PN2p/P1PBR2P/2NRKBn1 b q - 0 1"

NotM5Position = "2nrkbn1/pp1bp3/2qprp1p/6p1/4P2P/2NP1N2/PPP2PP1/RQB1KB1R w KQ - 0 1"
Zugzwang = "kbK5/pp6/1P6/8/8/8/8/R7 w - - 0 1"
StartingBoard = "rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1"
WMK2R = "8/8/8/8/8/4k3/6R1/6RK w - - 0 1"
WMK1REasy = "3k4/R7/8/4K3/8/8/8/8 w - - 0 1"
WMK1RMedium = "8/6K1/8/8/4k3/8/8/6R1 w - - 0 1"
WMK1RHard = "8/8/8/8/4k3/8/8/6RK w - - 0 1"
KingRoom = "8/8/8/8/3pp3/4k3/6R1/6RK w - - 0 1"

WMKQR = "8/8/8/3k4/8/8/8/5RQK w - - 0 1"

DKnightPromotionFork = "8/3KP1k1/5q2/8/8/8/8/8 w - - 0 1"

BTraxler = 'r1bqk2r/pppp1Npp/2n2n2/2b1p3/2B1P3/8/PPPP1PPP/RNBQK2R b KQkq - 0 1'
WAntiTraxler = 'r1bqk2r/pppp1ppp/2n2n2/2b1p1N1/2B1P3/8/PPPP1PPP/RNBQK2R w KQkq - 0 1'
Activity_vs_material = "rn2kbnr/1ppp1pp1/8/8/4P3/2P2N2/PB1P1PPP/RNQ1KB1R b KQkq - 0 5"

BM4SmutheredMate = "rnb2rk1/pp3ppp/1qpp4/8/2B1P1n1/3P1N2/PPP3PP/RNBQR2K b - - 6 10"
BM4SmutheredMate2 ="rnb2rk1/pp3ppp/1qpp4/8/2B3n1/3P1N2/PPP2PPP/RNBQR2K b - - 6 10"

## Imports

In [146]:
import random                                   
import numpy as np                              
import time                                     
import chess                                    

from ChessEnv.ZEIT4150.chess import ChessAgent, ChessGame
from IPython.display import display, HTML, clear_output
from chess import Move, polyglot, Board, Square, SquareSet, WHITE, BLACK

from math import log, sqrt
inf = float('inf')

## Board Scoring (Task 2)

The following functions evaluate the position on the board and produce a score between `-inf` and `+inf`. Positive scores favour the white player (i.e. `chess.WHITE` or `True`). Negative scores favour the black player (i.e. `chess.BLACK` or `False`)

In [147]:
# Attribution

# Portions of the board-scoring code in the next cell were generated with assistance from OpenAI ChatGPT (GPT-5.3-Codex), then reviewed and adapted by Brendon Bone.

# Reference: OpenAI (2026). ChatGPT. https://openai.com/models/chatgpt/

import operator
from functools import reduce

# AI-assisted block starts here (OpenAI ChatGPT, reviewed and adapted by Brendon Bone).
def prepBoard4Eval_2(b):
    """Prepares list of squares and attacking masks for white and black to be used in other evaluation functions."""
    b.squares = [SquareSet(b.occupied_co[BLACK]),
                 SquareSet(b.occupied_co[WHITE])]

    b.piece_masks = [b.pawns, b.knights, b.bishops, b.rooks, b.queens, b.kings]

    b.attacking_mask = [
        reduce(operator.__or__,
               (b.attacks_mask(sq) for sq in b.squares[BLACK]), 0),
        reduce(operator.__or__,
               (b.attacks_mask(sq) for sq in b.squares[WHITE]), 0)]
    return


def occupation_difference_2(b):
    """Difference in number of pieces on the board."""
    nWhiteSquares = bin(b.occupied_co[chess.WHITE]).count('1')
    nBlackSquares = bin(b.occupied_co[chess.BLACK]).count('1')
    return nWhiteSquares - nBlackSquares


def material_difference_2(b):
    """Difference in quality of pieces on the board."""
    piece_values = {
        chess.PAWN: 100,
        chess.KNIGHT: 320,
        chess.BISHOP: 330,
        chess.ROOK: 500,
        chess.QUEEN: 900,
        chess.KING: 0,
    }
    nWhite, nBlack = 0, 0
    for ptype, value in piece_values.items():
        nWhite += value * b.pieces_mask(ptype, chess.WHITE).bit_count()
        nBlack += value * b.pieces_mask(ptype, chess.BLACK).bit_count()
    return nWhite - nBlack


def bishop_difference_2(b):
    """Difference of bishops on the board with a bishop-pair bonus."""
    n_white_bishops = b.pieces_mask(chess.BISHOP, chess.WHITE).bit_count()
    n_black_bishops = b.pieces_mask(chess.BISHOP, chess.BLACK).bit_count()
    white_bishop_pair = 40 if n_white_bishops >= 2 else 0
    black_bishop_pair = 40 if n_black_bishops >= 2 else 0

    return (n_white_bishops - n_black_bishops) * 20 + white_bishop_pair - black_bishop_pair


def attack_difference_2(b):
    """Difference in attacking pieces for each attacked square."""
    nWhite, nBlack = 0, 0
    for sq in b.squares[chess.BLACK]:
        nWhite += len(b.attackers(chess.WHITE, sq))
    for sq in b.squares[chess.WHITE]:
        nBlack += len(b.attackers(chess.BLACK, sq))
    return nWhite - nBlack


def threat_difference_2(b):
    """Difference in pieces attacking the other colour."""
    nWhite = (b.attacking_mask[chess.WHITE] & b.occupied_co[chess.BLACK]).bit_count()
    nBlack = (b.attacking_mask[chess.BLACK] & b.occupied_co[chess.WHITE]).bit_count()
    return nWhite - nBlack


def center_difference_2(b):
    """Difference in number of pieces occupying the center."""
    center_squares = (chess.D4, chess.E4, chess.D5, chess.E5)
    nWhite, nBlack = 0, 0
    for sq in center_squares:
        piece = b.piece_at(sq)
        if piece is None:
            continue
        if piece.color == chess.WHITE:
            nWhite += 1
        else:
            nBlack += 1
    return nWhite - nBlack


def development_difference_2(b):
    """Opening feature: developed minor pieces difference."""
    white_home = {
        chess.B1: chess.KNIGHT,
        chess.G1: chess.KNIGHT,
        chess.C1: chess.BISHOP,
        chess.F1: chess.BISHOP,
    }
    black_home = {
        chess.B8: chess.KNIGHT,
        chess.G8: chess.KNIGHT,
        chess.C8: chess.BISHOP,
        chess.F8: chess.BISHOP,
    }

    white_undeveloped = sum(
        1 for sq, ptype in white_home.items()
        if (piece := b.piece_at(sq)) is not None and piece.color == chess.WHITE and piece.piece_type == ptype
    )
    black_undeveloped = sum(
        1 for sq, ptype in black_home.items()
        if (piece := b.piece_at(sq)) is not None and piece.color == chess.BLACK and piece.piece_type == ptype
    )

    return (4 - white_undeveloped) - (4 - black_undeveloped)


def passed_pawn_difference_2(b):
    """Reward passed pawns, scaled by advancement."""
    def _score(color):
        total = 0
        opp = not color
        opp_pawns = list(SquareSet(b.pieces_mask(chess.PAWN, opp)))
        for sq in SquareSet(b.pieces_mask(chess.PAWN, color)):
            f = chess.square_file(sq)
            r = chess.square_rank(sq)
            blocking = False
            for psq in opp_pawns:
                pf = chess.square_file(psq)
                pr = chess.square_rank(psq)
                if abs(pf - f) <= 1:
                    if color == chess.WHITE and pr > r:
                        blocking = True
                        break
                    if color == chess.BLACK and pr < r:
                        blocking = True
                        break
            if not blocking:
                adv = r if color == chess.WHITE else (7 - r)
                total += 8 + adv * 4
        return total

    return _score(chess.WHITE) - _score(chess.BLACK)


def hanging_piece_difference_2(b):
    """Penalize undefended non-pawn pieces (hanging pieces)."""
    piece_weights = {
        chess.KNIGHT: 14,
        chess.BISHOP: 14,
        chess.ROOK: 22,
        chess.QUEEN: 35,
    }

    def _score(color):
        total = 0
        for ptype, w in piece_weights.items():
            for sq in SquareSet(b.pieces_mask(ptype, color)):
                if len(b.attackers(color, sq)) == 0:
                    total -= w
        return total

    return _score(chess.WHITE) - _score(chess.BLACK)


def checkmate_2(b):
    """Returns a score for checkmate."""
    if not b.is_checkmate():
        return 0
    return -1 if b.turn == chess.WHITE else 1


def check_2(b):
    """Return score for check."""
    if not b.is_check():
        return 0
    return -1 if b.turn == chess.WHITE else 1


def isDraw_2(b):
    return b.can_claim_draw()


def draw_score_2(b):
    """Small draw penalty to prefer winning lines over sterile draws."""
    return -40 if isDraw_2(b) else 0


def getGamePhase_2(b):
    """Return phase in [0, 100]: 100 opening, 0 endgame."""
    material_units = (
        b.pieces_mask(chess.KNIGHT, chess.WHITE).bit_count() +
        b.pieces_mask(chess.BISHOP, chess.WHITE).bit_count() +
        b.pieces_mask(chess.ROOK, chess.WHITE).bit_count() +
        b.pieces_mask(chess.QUEEN, chess.WHITE).bit_count() +
        b.pieces_mask(chess.KNIGHT, chess.BLACK).bit_count() +
        b.pieces_mask(chess.BISHOP, chess.BLACK).bit_count() +
        b.pieces_mask(chess.ROOK, chess.BLACK).bit_count() +
        b.pieces_mask(chess.QUEEN, chess.BLACK).bit_count()
    )
    return min(100, int((material_units / 16) * 100))


def evalBoard_2(b):
    """Optimized phase-aware board evaluation. Removed expensive king_mobility and legal move counting."""
    prepBoard4Eval_2(b)
    phase = getGamePhase_2(b)

    if phase >= 67:
        # OPENING: develop pieces, control center, avoid hanging pieces
        weights = {
            'occupation': 2,
            'center': 16,
            'bishop': 1,
            'material': 1,
            'attack': 5,
            'threat': 8,
            'development': 18,
            'passed_pawn': 2,
            'hanging': 10,
            'check': 40,
            'checkmate': 100000,
        }
    elif phase >= 34:
        # MIDDLEGAME: balanced tactical and strategic pressure
        weights = {
            'occupation': 2,
            'center': 10,
            'bishop': 1,
            'material': 1,
            'attack': 8,
            'threat': 10,
            'development': 6,
            'passed_pawn': 8,
            'hanging': 14,
            'check': 60,
            'checkmate': 100000,
        }
    else:
        # ENDGAME: convert passed pawns into wins, material is king
        weights = {
            'occupation': 1,
            'center': 4,
            'bishop': 1,
            'material': 1,
            'attack': 5,
            'threat': 7,
            'development': 0,
            'passed_pawn': 20,
            'hanging': 8,
            'check': 40,
            'checkmate': 100000,
        }

    base_score = (
        weights['occupation'] * occupation_difference_2(b) +
        weights['center'] * center_difference_2(b) +
        weights['bishop'] * bishop_difference_2(b) +
        weights['material'] * material_difference_2(b) +
        weights['attack'] * attack_difference_2(b) +
        weights['threat'] * threat_difference_2(b) +
        weights['development'] * development_difference_2(b) +
        weights['passed_pawn'] * passed_pawn_difference_2(b) +
        weights['hanging'] * hanging_piece_difference_2(b) +
        weights['check'] * check_2(b) +
        weights['checkmate'] * checkmate_2(b)
    )

    return (0 if isDraw_2(b) else base_score) + draw_score_2(b)
# AI-assisted block ends here.


## Move Ranking (Task 3)

In [148]:
# Attribution:
# This move-ordering heuristic was drafted with AI assistance (OpenAI ChatGPT via GitHub Copilot),
# then reviewed and adapted for this notebook's alpha-beta search requirements.

# Lightweight piece values for MVV-LVA (Most Valuable Victim - Least Valuable Attacker)
_PIECE_VALUE = (0, 100, 320, 330, 500, 900, 0)  # index by chess piece type (0..6)

# Promotion bonuses (higher = searched earlier)
_PROMO_BONUS = {
    chess.QUEEN: 9000,
    chess.ROOK: 5000,
    chess.BISHOP: 3300,
    chess.KNIGHT: 3200,
}

# Small positional bonuses for quiet move ordering
_CENTER_4 = {chess.D4, chess.E4, chess.D5, chess.E5}
_CENTER_16 = {
    chess.C3, chess.D3, chess.E3, chess.F3,
    chess.C4, chess.D4, chess.E4, chess.F4,
    chess.C5, chess.D5, chess.E5, chess.F5,
    chess.C6, chess.D6, chess.E6, chess.F6,
}

def _capture_victim_square(b, mv):
    """Return the square of the captured piece (handles en passant)."""
    if b.is_en_passant(mv):
        file_ = chess.square_file(mv.to_square)
        rank_ = chess.square_rank(mv.from_square)
        return chess.square(file_, rank_)
    return mv.to_square


def rankMove(b, mv):
    """
    Fast move-ordering score (higher is better).
    Prioritises tactical forcing moves for stronger alpha-beta pruning.
    Adds an extra bonus for capturing hanging (undefended) pieces.
    """
    score = 0
    attacker = b.piece_type_at(mv.from_square) or chess.PAWN

    # 1) Captures: MVV-LVA + hanging-piece bonus
    if b.is_capture(mv):
        victim_sq = _capture_victim_square(b, mv)
        victim_piece = b.piece_at(victim_sq)
        victim = chess.PAWN if victim_piece is None else victim_piece.piece_type

        score += 10000 + 16 * _PIECE_VALUE[victim] - _PIECE_VALUE[attacker]

        # Bonus when captured piece is undefended by its own side.
        victim_color = not b.turn
        if len(b.attackers(victim_color, victim_sq)) == 0:
            score += 2500 + 2 * _PIECE_VALUE[victim]

    # 2) Promotions
    if mv.promotion:
        score += _PROMO_BONUS.get(mv.promotion, 0)

    # 3) Checks and castling (forcing / king safety)
    if b.gives_check(mv):
        score += 3000
    if b.is_castling(mv):
        score += 1200

    # 4) Pawn advances / irreversible progress
    if b.is_zeroing(mv):
        score += 150

    # 5) Small quiet-move positional ordering
    to_sq = mv.to_square
    if to_sq in _CENTER_4:
        score += 40
    elif to_sq in _CENTER_16:
        score += 20

    # 6) Mild penalty for non-castling king moves in non-tactical cases
    if attacker == chess.KING and not b.is_castling(mv):
        score -= 50

    return score


## Example: Random Agent (Unchanged)

In [149]:
class RandomChessAgent(ChessAgent):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        pass
    
    def calcMoves(self, board):
        # Always wrap legal_moves into a list to 
        # unpack moves. Otherwise, it returns 
        # a generator.
        legalMoves = list(board.legal_moves)
        
        N = 2
        
        myMoves = []
        myMoves = np.random.choice(legalMoves, N).tolist()
        # Draw list of moves on the board (blue)
        self.drawMyMoves(myMoves)
        
        # Log relevant info per agent
        self.log2Agent(f"INFO: myMoves={[board.san(mv) for mv in myMoves]}", "green")
        
        # Draw a minitutre version of this board
        # You can use this to visualise current scenario
        # your agent is investigating.
        self.drawBoard(board, "Board1")
        
        # Log info to the margin of the board. This is a shared 
        # logging space between this agent, the opponent agent 
        # and the board itself.
        # self.log2Margin(f"This is an INFO msg!!")
        for mv in myMoves:
            
            # Apply action to the board
            board.push(mv)
            
            # Query leagal moves after action`mv` applied
            legalMoves = list(board.legal_moves)
            
            if len(legalMoves) >= N:
                oppMoves = np.random.choice(legalMoves, 2).tolist()
                
                # Log relevant info per agent
                self.log2Agent(f"INFO: oppMoves={[board.san(mv) for mv in oppMoves]}", "red")
                
                # Draw list of opponent moves on the board (red)
                self.drawOpponentMoves(oppMoves)
            board.pop()
            pass
        
        # Draw a mask board showing king's reach
        self.drawBoard(board.attacks(board.king(board.turn)), 
                       "King")
        
        # Draw a mask board showing opponent's king's reach
        self.drawBoard(board.attacks(board.king(not board.turn)), 
                       "Other King")
        
        self.drawBoard(board, "Mask 3")
        self.drawBoard(board.attackers(chess.WHITE, chess.F5), 
                       "f5 Attackers")
        
        self.drawBoard(board.attackers(chess.WHITE, chess.E5), 
                       "e5 Attackers")
        
        self.drawBoard(board.attackers(chess.WHITE, chess.D5), 
                       "d5 Attackers")
        
        self.drawBoard(chess.SquareSet(chess.BB_LIGHT_SQUARES & chess.BB_RANK_3), 
                       "3rd Rank Light Squares")
        
        # Scoring function check base class for more info 
        score = self.scoreFn(self.board)
        self.log2Agent(f"{score=}", "black")
        
        # A sleeping timer to slow things down (only releven when visualising random agents)
        time.sleep(self.dtime)
        return myMoves[-1]
        
    def getAction(self, board):
        return self.calcMoves(board)
    pass

## MinMax Adversarial state space search algorithm (Task 1a)

To implement MinMax this video: https://www.youtube.com/watch?v=l-hh51ncgDI

Along with this Github reposity: https://github.com/nadeem4/chess_engine_using_python/tree/main
Inspired my coding

These sources are used for both Task 1A and 1B

In [150]:
class MinMaxAgent(ChessAgent):
    # This was copied from the random agent and adapted to implement a minmax search.
    def __init__(self, maxDepth=2, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.maxDepth = maxDepth
        self.last_search_time = 0.0  # seconds

    # MinMax algorithm implementation
    def _minmax(self, board, depth, maximizing_white):
        if depth == 0 or board.is_game_over(claim_draw=True):
            return self.scoreFn(board), None

        # Find all legal moves
        legal_moves = list(board.legal_moves)

        # If trying to maximise white's advantage, we want to find the move with the highest score.
        if maximizing_white:
            
            # Initialize the best score and move (anything is better than -inf)
            best_score = -inf
            
            for mv in legal_moves:
                board.push(mv)
                score, _ = self._minmax(board, depth - 1, False)
                board.pop()

                if score > best_score:
                    best_score = score
                    best_move = mv
            return best_score, best_move

        # If trying to maximise black's advantage, we want to find the move with the lowest score.
        best_score = inf
        for mv in legal_moves:
            board.push(mv)
            score, _ = self._minmax(board, depth - 1, True)
            board.pop()

            if score < best_score:
                best_score = score
                best_move = mv
        return best_score, best_move

    
    def calcMoves(self, board):
        maximizing_white = (board.turn == chess.WHITE)

        t0 = time.perf_counter()
        score, best_move = self._minmax(board, self.maxDepth, maximizing_white)
        self.last_search_time = time.perf_counter() - t0
        
        # Draw the next move on the board (Implemented from the random agent)
        self.drawMyMoves([best_move])
        
        # Troubleshooting data
        self.log2Agent(
            f"INFO: MinMax depth={self.maxDepth}, move={board.san(best_move)}, "
            f"score={score}, time={self.last_search_time:.3f}s",
            "green",
        )

        # Pause for a moment directly copied from the random agent
        time.sleep(self.dtime)
        return best_move

    # Iniates the decision to make the next move
    def getAction(self, board):
        return self.calcMoves(board)




## Enhanced MinMax by using AlphaBeta pruning (Task 1B)

The same resources used in Task 1A was used to assist implementing the AlphaBeta pruning

In [151]:
class MinMaxAgent_AlphaBetaPruning(ChessAgent):
    def __init__(self, maxDepth=2, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.maxDepth = maxDepth
        self.last_search_time = 0.0  # seconds

    def _alphabeta(self, board, depth, alpha, beta, maximizing_white):
        if depth == 0 or board.is_game_over(claim_draw=True):
            return self.scoreFn(board), None

        legal_moves = list(board.legal_moves)

        if maximizing_white:
            best_score = -inf
            best_move = legal_moves[0]

            for mv in legal_moves:
                board.push(mv)
                score, _ = self._alphabeta(board, depth - 1, alpha, beta, False)
                board.pop()

                if score > best_score:
                    best_score = score
                    best_move = mv

                alpha = max(alpha, best_score)
                if alpha >= beta:
                    break # No need to explore further

            return best_score, best_move

        best_score = inf
        best_move = legal_moves[0]



        for mv in legal_moves:
            board.push(mv)
            score, _ = self._alphabeta(board, depth - 1, alpha, beta, True)
            board.pop()

            if score < best_score:
                best_score = score
                
                best_move = mv

            beta = min(beta, best_score)
            if alpha >= beta:
                break # No need to explore further

        return best_score, best_move

    def calcMoves(self, board):
        maximizing_white = (board.turn == chess.WHITE)

        t0 = time.perf_counter() # Start timer
        score, best_move = self._alphabeta(board, self.maxDepth, -inf, inf, maximizing_white)
        self.last_search_time = time.perf_counter() - t0 # End timer

        self.drawMyMoves([best_move])
        self.log2Agent(
            f"INFO: AlphaBeta depth={self.maxDepth}, move={board.san(best_move)}, "
            f"score={score}, time={self.last_search_time:.3f}s",
            "green",
        )

        time.sleep(self.dtime)
        return best_move

    def getAction(self, board):
        return self.calcMoves(board)




## enhanced Alpha-Beta pruning using move ordering (Task 1C)

In [152]:
class AlphaBeta_enhanced(ChessAgent):
    """AlphaBeta pruning with move ordering for faster cutoffs."""
    def __init__(self, maxDepth=2, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.maxDepth = maxDepth
        self.last_search_time = 0.0  # seconds

    def _alphabeta(self, board, depth, alpha, beta, maximizing_white):
        if depth == 0 or board.is_game_over(claim_draw=True):
            return self.scoreFn(board), None

        legal_moves = list(board.legal_moves)
        
        # Move ordering: Sorts the legal_moves using the less computationally expensive rankMove heuristic to prioritise promising moves for faster alpha-beta cutoffs.
        legal_moves.sort(key=lambda mv: rankMove(board, mv), reverse=True)

        if maximizing_white:
            best_score = -inf
            best_move = legal_moves[0]

            for mv in legal_moves:
                board.push(mv)
                score, _ = self._alphabeta(board, depth - 1, alpha, beta, False)
                board.pop()

                if score > best_score:
                    best_score = score
                    best_move = mv

                alpha = max(alpha, best_score)
                if alpha >= beta:
                    break # No need to explore further

            return best_score, best_move

        best_score = inf
        best_move = legal_moves[0]

        for mv in legal_moves:
            board.push(mv)
            score, _ = self._alphabeta(board, depth - 1, alpha, beta, True)
            board.pop()

            if score < best_score:
                best_score = score
                best_move = mv

            beta = min(beta, best_score)
            if alpha >= beta:
                break # No need to explore further

        return best_score, best_move

    def calcMoves(self, board):
        maximizing_white = (board.turn == chess.WHITE)

        t0 = time.perf_counter() # Start timer
        score, best_move = self._alphabeta(board, self.maxDepth, -inf, inf, maximizing_white)
        self.last_search_time = time.perf_counter() - t0 # End timer

        self.drawMyMoves([best_move])
        self.log2Agent(
            f"INFO: AlphaBeta+MoveOrder depth={self.maxDepth}, move={board.san(best_move)}, "
            f"score={score}, time={self.last_search_time:.3f}s",
            "green",
        )

        time.sleep(self.dtime)
        return best_move

    def getAction(self, board):
        return self.calcMoves(board)
    
    
    

## Reduced redundant calculations using Zobrist hashed transposition tables (Task 1d)

This is based on the Transpoition and Memoisation table using in the WK04.2-L-Adversarial Search and Games-2

In [153]:
# AlphaBeta pruning with move ordering + Zobrist transposition table

class AlphaBeta_Zobrist(ChessAgent):
    
    # Once all the optimisations are in place, we can increase the search depth to 4 for stronger play.
    def __init__(self, maxDepth=4, *args, **kwargs): 
        super().__init__(*args, **kwargs)
        self.maxDepth = maxDepth
        self.last_search_time = 0.0  # seconds

        # Zobrist hashing lookup table for memoization of previously evaluated positions.        
        self.TranspositionTable = {}
        # Example: TranspositionTable[hash_key] = {"board": fen, "score": score, "alpha": alpha, "beta": beta, "at_depth": depth, "depth_searched": ply, "best_move_uci": "e2e4"}
        
    # Generate a unique hash for the current board position using Zobrist hashing.
    def _hash_board(self, board):
        return str(polyglot.zobrist_hash(board))

    def _alphabeta(self, board, depth, alpha, beta, maximizing_white, ply=0):
        if depth == 0 or board.is_game_over(claim_draw=True):
            return self.scoreFn(board), None
        
        key = self._hash_board(board)
        entry = self.TranspositionTable.get(key)

        # This position has not been evaluated before, move on and evaluate it as normal
        if entry is None:
            pass 
        
        # This position was evaluated before, but at a shallower depth.
        elif entry["at_depth"] < depth:  
            pass
        
        # This position was evaluated before, but at a deeper depth. 
        elif not (entry["alpha"] <= alpha and entry["beta"] >= beta):
            pass
        
        # This position was evaluated before
        else:
        
            uci = entry["best_move_uci"] # Find the best move from the transposition table entry
            best_move = chess.Move.from_uci(uci) # Convert the UCI string back to a chess.Move object
            return entry["score"], best_move # Return the stored score and best move for this position without re-evaluating it
                
        

        # Find all legal moves
        legal_moves = list(board.legal_moves)
        
        # Rough move ordering using the rankMove heuristic 
        legal_moves.sort(key=lambda mv: rankMove(board, mv), reverse=True)

        alpha_in, beta_in = alpha, beta

        if maximizing_white:
            best_score = -inf
            best_move = legal_moves[0]

            for mv in legal_moves:
                board.push(mv)
                score, _ = self._alphabeta(board, depth - 1, alpha, beta, False, ply + 1)
                board.pop()

                if score > best_score:
                    best_score = score
                    best_move = mv

                alpha = max(alpha, best_score)
                if alpha >= beta:
                    break  # No need to explore further

        else:
            best_score = inf
            best_move = legal_moves[0]

            for mv in legal_moves:
                board.push(mv)
                score, _ = self._alphabeta(board, depth - 1, alpha, beta, True, ply + 1)
                board.pop()

                if score < best_score:
                    best_score = score
                    best_move = mv

                beta = min(beta, best_score)
                if alpha >= beta:
                    break  # No need to explore further

        # Memoize this position using the requested board+search metadata.
        self.TranspositionTable[key] = {
            "board": board.fen(),
            "score": best_score,
            "alpha": alpha_in,
            "beta": beta_in,
            "at_depth": depth,
            "depth_searched": ply,
            "best_move_uci": best_move.uci(),
        }

        return best_score, best_move

    def calcMoves(self, board):
        maximizing_white = (board.turn == chess.WHITE)

        # Fresh table each root search
        self.TranspositionTable.clear()

        t0 = time.perf_counter() # Start timer
        score, best_move = self._alphabeta(board, self.maxDepth, -inf, inf, maximizing_white, 0)
        self.last_search_time = time.perf_counter() - t0 # End timer

        self.drawMyMoves([best_move])
        self.log2Agent(
            f"INFO: AlphaBeta+Zobrist depth={self.maxDepth}, move={board.san(best_move)}, "
            f"score={score}, time={self.last_search_time:.3f}s, ",
        )

        time.sleep(self.dtime)
        return best_move

    def getAction(self, board):
        return self.calcMoves(board)

## Game Play

### Test Cases

In [154]:
# Starting board. This is the standard chess game. White moves 1st. 
StartingBoard = "rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1"

# White is trapped in a dicovery check by the knight and the queen. Black's knight 
# forks the king and the queen on F2 forcing the white king to H1. Black then has to
# reject taking the queen on D1 and instead deliver a double check (queen to assist) 
# which forces the white king hide in the corner. Black will then sacrifice the queen 
# on G1 which forces the white Rook to take and trapping its own king even further. 
# Finally, Black delivers check mate with Nf2#.
# This puzzle measures the long term planning which guarantees a win if the agent 
# ignores material gain (ignoring the forked queen on D1) and even sacrificing its 
# own queen on G1.
# 
# Solution (Black checkmates in 4 moves): 10... Nf2+ 11. Kg1 Nh3+ 12. Kh1 Qg1+ 13. Rxg1 Nf2#
BM4SmutheredMate = "rnb2rk1/pp3ppp/1qpp4/8/2B1P1n1/3P1N2/PPP3PP/RNBQR2K b - - 6 10"

# This puzzle, the white knight has to clear the way to the queen to deliver checkmate 
# on H5. The knight has to choose between taking the pawn on G5 or just move to E5. Both 
# moves clears the way to the queen but Ne5 is the correct move because it blocks D7. 
# If not Ne5, Black can clear the D7 square after sacrificing the rook on E6 with a check.
# 
# Solution (White checkmates in 5 moves): 1. Ne5 g4 2. Qxg4 Qxc3+ 3. bxc3 fxe5 4. Qh5+ Rg6 5. Qxg6#
WM5Position = "2nrkbn1/pp1bp3/2qprp1p/6p1/4P2P/2NP1N2/PPP2PP1/R1BQKB1R w KQ - 0 9"

# This puzzle measures precision. The black king is trapped at A8. If White loses 
# the rook, the game ends in a draw. It is white's turn and Black is counting on this 
# to ease the pressure. White needs to waste a move and be forcing at the same time. 
# The wasted move forces Black to chose between two very bad moves and losing the game.
# This tractic is what is known as "Zugzwang". The solution is to move the rook to A6
# forcing Black to take with the pawn and thus allows White to push the B6 pawnto B7 
# and checkmates Black.
# 
# Solution (White checkmates in 2 moves): 53. Ra6 bxa6 54. b7#


BM2Zugzwang = "kbK5/pp6/1P6/8/8/8/8/R7 w - - 0 53" # SUCCESSFUL

# This puzzle measures precision. There is one order of moves to solve this puzzle. 
# The white rook on E1 (not C1) must check the black king. The the knight delivers a
# a check on F4. Finally, the white rook on C1 delivers a check mate on c3.
# 
# Solution (White checkmates in 3 moves): 30. Red1+ Ke2 31. Nf4+ Ke3 32. Rc3#


WM3MateNet = "3rr3/p5bp/2nB2p1/2PN4/3P4/3k1P2/P5PP/2R1R1K1 w - - 4 30" # SUCCESSFUL

# End-game puzzles aim to measure the efficacy of transposition table and the scoring 
# function. White can win easily with a king and a rook. Black is aiming to draw the 
# game by repetition or 50 moves rules. The white King must move to G6. Black can only 
# go to G8. Finally the white rook delivers mate on A1.
# 
# Solution (White checkmates in 2 moves): 62. Kg6 Kg8 63. Ra8#
WM2EndGameKingRook = "7k/R7/8/5K2/8/8/8/8 w - - 0 62" #  SUCCESSFUL

# End-game puzzles aim to measure the efficacy of transposition table and the scoring 
# function. White can win easily with a 2 rooks or a rook and a queen. Black is aiming
# to draw the game by repetition or 50 moves rules. The white rooks can perform a 
# ladder mate with rooks checking the black king on A6, B7 and delivers mate on A8.
# 
# Solution (White checkmates in 3 moves): 62. Ra6+ Kh7 63. Rb7+ Kh8 64. Ra8#
WM3EndGame2RooksLadderMate = "8/8/6k1/1R6/8/8/8/R2K4 w - - 0 62" # SUCCESSFUL

# This end-game puzzle dictates on white to under promote the pawn to a bishop to avoid
# a stalemate. White pushes the pawn to D8. Black's only moveable piece is the knight
# and the correct move is knight to F8. Now if white promotes to a queen or a rook, the
# black knight is pinned and thus black has no legal moves but no in check (i.e. stalemate)
# the game is drawn. Thus white must promote to a bishop on D8 and thus the black knight is 
# available to move (i.e. stalemate avoided) and the game continues with mate in 2.
# 
# Solution (White checkmates in 4 moves): 64. d7 Nf8 65. d8=B Ne6 66. Bf6+ Ng7 67. Bxg7#
WM4UnderPromotion = "7k/5B1n/3P3K/8/8/8/8/8 w - - 0 64"



### Game Play

In [155]:
import io

white = AlphaBeta_Zobrist(name="White", scoreFn=evalBoard_2, dtime=0, maxDepth=4)
black = AlphaBeta_Zobrist(name="Black", scoreFn=evalBoard_2, dtime=0, maxDepth=4)

game = ChessGame(fen=WM4UnderPromotion, vizSize=350)
clear_output(wait=True)

for i in range(1):
    game.addGameInfo(f"Game {i:03d}: {white.name} vs. {black.name}")
    game.playGame(white, black, visualise=True)

    pgn = game.saveGame(f"Game__{i} - {white.name} vs. {black.name}.pgn")

    pgn_game = chess.pgn.read_game(io.StringIO(pgn))
    moves = list(pgn_game.mainline_moves())
    white_moves = (len(moves) + 1) // 2
    black_moves = len(moves) // 2

    print(f"Game {i:03d} move count -> White: {white_moves}, Black: {black_moves}, Total: {len(moves)}")

    clear_output(wait=True)
    game.resetGame()
    

Game 000 move count -> White: 57, Black: 57, Total: 114


## Task 4: Explanation

### How the AI agent is able to play the game of chess 4A

The agent is able to play games of chess by encoding the position of each piece using an evaluation function and continuously maximising it until it reaches the end state - a checkmate. In my code the function `evalBoard_2(b)` evaluates the board and a MinMax algorithm is used to find the next action that gets it closer to the end state.

The MinMax is a simple but effective search algorithm used to maximise the evaluation function in its favor (high number for white and low for black). The agent considers every possible action it and the adversary can take to maximise their evaluation function and returns the action it should get closer to win.

If unlimited computing power was available, the MinMax algorithm would be expanded indefinitely until it find the exact action it should take to win the game is found. However, after each move the number of potential moves increase significantly, and therfore expanding every node until a solution is found is not always possible. Therefore, the MinMax depth is usually limited (in this assignment maxDepth=2 and maxDepth=2 for `MinMaxAgent` and `AlphaBeta_Zobrist` respectfully). This means the action it takes now is maximising the value evaluation function 2-4 moves ahead rather then chosing the action that will the win. This means the agents performance is reliant on the accuracy of the evaluation functions ability to quantify being closer to an end state.


The function `evalBoard_2(b)` is relatively computationally heavy (compared to the `rankMove(b, mv)` function). Usually the evaluation function is created with the assistance of a professional in the field but as I am not a professional, generative AI was used to assist in the development on the evaluation function. These are the main considerations the evaluation function used:



- Material
  - Who has more valuable pieces

- Piece count
  - Who has more pieces left on the board

- Center control
  - Who occupies important central squares

- Attacking pressure
  - Who is attacking more and creating more threats

- King condition
  - Checkmate
  - Check
  - King mobility
  - King safety

- Development
  - Who has developed pieces better in the opening

- Pawn structure
  - Healthy pawn chains and fewer pawn weaknesses

- Game phase
  - Opening
  - Middlegame
  - Endgame
  - Different features matter more in different phases

- Draw state
  - Drawn positions are penalised

Final score
- Positive = good for White
- Negative = good for Black


Attribution: The following summary of the main board-evaluation concepts was generated with assistance from OpenAI’s ChatGPT
Reference:
OpenAI (2026). ChatGPT. OpenAI. https://openai.com/models/chatgpt/
Board evaluation main concepts






### Agent Explanation (Task 4B)

Each iteration of the MinMax agent was kept throughout the assignment. 

